# Plot timeseries

In [2]:
import os
import xarray as xr
import matplotlib.pyplot as plt


# ファイル読み込み
domain_id = 'Bangkok'
wrf_out_d1 = f'../../sample_data/{domain_id}/wrfout_d01_2025-01-02_00:00:00'
geo_em_d2 = f'../../sample_data/{domain_id}/wrfout_d02_2025-01-02_00:00:00'
geo_em_d3 = f'../../sample_data/{domain_id}/wrfout_d03_2025-01-01_00:00:00'

ds3 = xr.open_dataset(geo_em_d3, engine='netcdf4')
ds3

<xarray.Dataset> Size: 1GB
Dimensions:                (Time: 24, south_north: 99, west_east: 99,
                            bottom_top: 44, bottom_top_stag: 45,
                            soil_layers_stag: 4, west_east_stag: 100,
                            south_north_stag: 100, seed_dim_stag: 8)
Coordinates:
    XLAT                   (Time, south_north, west_east) float32 941kB ...
    XLONG                  (Time, south_north, west_east) float32 941kB ...
    XTIME                  (Time) datetime64[ns] 192B ...
    XLAT_U                 (Time, south_north, west_east_stag) float32 950kB ...
    XLONG_U                (Time, south_north, west_east_stag) float32 950kB ...
    XLAT_V                 (Time, south_north_stag, west_east) float32 950kB ...
    XLONG_V                (Time, south_north_stag, west_east) float32 950kB ...
Dimensions without coordinates: Time, south_north, west_east, bottom_top,
                                bottom_top_stag, soil_layers_stag,
                                west_east_stag, south_north_stag, seed_dim_stag
Data variables: (12/210)
    Times                  (Time) |S19 456B ...
    LU_INDEX               (Time, south_north, west_east) float32 941kB ...
    ZNU                    (Time, bottom_top) float32 4kB ...
    ZNW                    (Time, bottom_top_stag) float32 4kB ...
    ZS                     (Time, soil_layers_stag) float32 384B ...
    DZS                    (Time, soil_layers_stag) float32 384B ...
    ...                     ...
    PCB                    (Time, south_north, west_east) float32 941kB ...
    PC                     (Time, south_north, west_east) float32 941kB ...
    LANDMASK               (Time, south_north, west_east) float32 941kB ...
    LAKEMASK               (Time, south_north, west_east) float32 941kB ...
    SST                    (Time, south_north, west_east) float32 941kB ...
    SST_INPUT              (Time, south_north, west_east) float32 941kB ...
Attributes: (12/136)
    TITLE:                            OUTPUT FROM WRF V4.6.1 MODEL
    START_DATE:                      2025-01-01_00:00:00
    SIMULATION_START_DATE:           2025-01-01_00:00:00
    WEST-EAST_GRID_DIMENSION:        100
    SOUTH-NORTH_GRID_DIMENSION:      100
    BOTTOM-TOP_GRID_DIMENSION:       45
    ...                              ...
    ISLAKE:                          21
    ISICE:                           15
    ISURBAN:                         13
    ISOILWATER:                      14
    HYBRID_OPT:                      2
    ETAC:                            0.2

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import xarray as xr

# ----------------------
# USER SETTINGS
# ----------------------
domain_id = 'Bangkok'
station_id = "THM00048419"
dataset    = "GHCNh"         # used to form the obs filename
year       = 2025
wrf_glob   =  f"../../sample_data/{domain_id}/wrfout_d03_*"  # pattern for wrfout files in the working dir
stations_csv = "../../sample_data/point_data/GHCNh/summaries/stations_summary.csv"

# Which WRF variables to pull + how to map them to obs column names
# (left = WRF var in wrfout; right = column name in the obs CSV)
variable_map = {
    "T2": "temperature",           # K in WRF (convert to °C below)
    "Q2": "specific_humidity",     # kg/kg
    "U10": "u10",                  # m/s
    "V10": "v10",                  # m/s
    "RAINNC": "precip"             # mm (cumulative); keep as-is or differencing later
}

# ----------------------
# 1) Read station lat/lon
# ----------------------
st = pd.read_csv(stations_csv)

# Try common column name variants
lat_col = next(c for c in st.columns if c.lower() in ("lat","latitude"))
lon_col = next(c for c in st.columns if c.lower() in ("lon","longitude","lng"))
id_col  = next(c for c in st.columns if c.lower() in ("station_id","id","station", "ghcn_id"))

row = st.loc[st[id_col].astype(str) == str(station_id)]
if row.empty:
    raise ValueError(f"station_id={station_id} not found in {stations_csv}")
lat0 = float(row.iloc[0][lat_col])
lon0 = float(row.iloc[0][lon_col])

# ----------------------
# 2) Open WRF output(s) and locate nearest grid cell
# ----------------------
wrf_files = sorted(glob.glob(wrf_glob))
if not wrf_files:
    raise FileNotFoundError(f"No files matched pattern: {wrf_glob}")

wrf_file = wrf_files[0]
# Combine by coordinates across multiple wrfout files (daily/hourly chunks, etc.)
# ds = xr.open_mfdataset(wrf_files, combine="by_coords", decode_times=False)
ds = xr.open_mfdataset(wrf_file,  decode_times=False)

# Get 2D lat/lon (take first time slice if Time exists)
def _pick_2d(arr):
    return arr.isel(Time=0) if "Time" in arr.dims else arr

XLAT  = _pick_2d(ds["XLAT"])
XLONG = _pick_2d(ds["XLONG"])

# Nearest-neighbor (on lat/lon; good enough for small spatial scales)
dist2 = (XLAT - lat0)**2 + (XLONG - lon0)**2
j, i = np.unravel_index(np.argmin(dist2.values), dist2.shape)

# ----------------------
# 3) Build WRF time index and extract time series
# ----------------------
def wrf_timeindex(dset: xr.Dataset) -> pd.DatetimeIndex:
    # Prefer CF-like decoded time if available; otherwise decode WRF 'Times'
    if "XTIME" in dset.variables:
        # minutes since start time → we still need an absolute time; many WRF builds include 'SIMULATION_START_DATE'
        # If absent, fall back to 'Times'
        try:
            start = pd.to_datetime(str(dset.attrs.get("SIMULATION_START_DATE")))
            return (pd.to_timedelta(dset["XTIME"].values, "m") + start).to_pydatetime()
        except Exception:
            pass
    if "Times" in dset.variables:
        # 'Times' is (Time, DateStrLen) char array like b'YYYY-MM-DD_HH:MM:SS'
        tchars = dset["Times"].load().values  # (n, m) bytes
        tstrings = ["".join(char.decode() for char in row) for row in tchars]
        return pd.to_datetime(tstrings, format="%Y-%m-%d_%H:%M:%S")
    # Fallback: try 'Time' as datetime64 if present
    if "Time" in dset.dims:
        try:
            return pd.to_datetime(dset["Time"].values)
        except Exception:
            pass
    raise RuntimeError("Could not construct a proper time index from wrfout.")

time_index = pd.DatetimeIndex(wrf_timeindex(ds))

wrf_df = pd.DataFrame(index=time_index)

# Extract and lightly post-process each requested variable (if present)
for wrf_var, obs_col in variable_map.items():
    if wrf_var in ds.variables:
        series = ds[wrf_var].isel(south_north=j, west_east=i)
        # If variable has a Time dimension, squeeze it to 1D; decode to numpy
        if "Time" in series.dims:
            s = pd.Series(series.values, index=time_index, name=obs_col)
        else:
            # No time dimension → not a time series; skip
            continue
        # Unit tweaks (examples)
        if wrf_var == "T2":
            s = s - 273.15  # Kelvin → °C
        wrf_df[obs_col + "_wrf"] = s

# Optional: if RAINNC is cumulative and you want rate per time step:
# if "precip_wrf" in wrf_df.columns:
#     wrf_df["precip_wrf"] = wrf_df["precip_wrf"].diff().clip(lower=0)

# Keep only the target year (if desired)
wrf_df = wrf_df[wrf_df.index.year == int(year)]

# ----------------------
# 4) Load observations and align
# ----------------------
obs_path = f"{dataset}_{station_id}_{year}.csv"
if not os.path.exists(obs_path):
    raise FileNotFoundError(f"Observation file not found: {obs_path}")

# Try to parse time; guess time column
obs = pd.read_csv(obs_path)
# Find a likely datetime column
time_col_candidates = [c for c in obs.columns if c.lower() in ("time","datetime","date","timestamp")]
if time_col_candidates:
    tcol = time_col_candidates[0]
    obs[tcol] = pd.to_datetime(obs[tcol])
    obs = obs.set_index(tcol)
else:
    # If no explicit time column, attempt to parse the index
    try:
        obs.index = pd.to_datetime(obs.index)
    except Exception:
        raise ValueError("Could not find/parse a datetime column in observations.")

# Inner-join on timestamp
df_merged = wrf_df.join(obs, how="inner", rsuffix="_obsraw")

# ----------------------
# 5) Quick comparison stats (bias/RMSE) for mapped variables that exist on both sides
# ----------------------
def stats_pair(wrf_name, obs_name, frame):
    a = frame[wrf_name].astype(float)
    b = frame[obs_name].astype(float)
    common = a.notna() & b.notna()
    if not common.any():
        return pd.Series({"n":0, "bias":np.nan, "rmse":np.nan, "corr":np.nan})
    diff = a[common] - b[common]
    return pd.Series({
        "n": common.sum(),
        "bias": diff.mean(),
        "rmse": np.sqrt((diff**2).mean()),
        "corr": a[common].corr(b[common])
    })

rows = []
for wrf_var, obs_col in variable_map.items():
    wrf_col = obs_col + "_wrf"
    if wrf_col in df_merged.columns and obs_col in df_merged.columns:
        rows.append(stats_pair(wrf_col, obs_col, df_merged).rename(obs_col))

comparison_stats = pd.DataFrame(rows)

# ----------------------
# 6) Show/save results
# ----------------------
print(f"Nearest WRF grid indices for station {station_id}: j={j}, i={i} (lat0={lat0}, lon0={lon0})")
print("\nPreview (merged):")
print(df_merged.head())

print("\nComparison stats (WRF vs OBS):")
print(comparison_stats)

# Optionally save outputs
df_merged.to_csv(f"wrf_vs_obs_{station_id}_{year}.csv")
comparison_stats.to_csv(f"wrf_vs_obs_stats_{station_id}_{year}.csv")


ValueError: Could not find any dimension coordinates to use to order the datasets for concatenation